# E-commerce Clickstream Analytics with PySpark
### A beginner-friendly end-to-end project (31 code steps)

**Business problem:** an online electronics store records every customer click (view, add to cart, purchase) but has not turned that data into decisions. In this project we use PySpark to find out **where the revenue comes from, where customers drop out, when they shop, which customers matter most, and whether a cart will turn into a purchase.**

| | |
|---|---|
| **Data** | `events.csv`: 885,129 events from 24 Sept 2020 to 28 Feb 2021 |
| **Columns** | `event_time`, `event_type` (view / cart / purchase), `product_id`, `category_id`, `category_code`, `brand`, `price`, `user_id`, `user_session` |
| **Tools** | PySpark (DataFrame and RDD), Spark MLlib, pandas, scikit-learn, matplotlib, Parquet |

## Project questions and where they are answered

| # | Business question | Steps |
|---|---|---|
| 1 | Is the data clean enough to analyse? | 5 to 10 |
| 2 | Where does revenue come from (categories, brands, products)? | 11 to 14 |
| 3 | How did sales and revenue change over time? | 15, 16 |
| 4 | Where do customers drop out (funnel, cart abandonment)? | 17 |
| 5 | When do customers shop? | 18 |
| 6 | Which customers matter most? | 19, 20 |
| 7 | Do low-level RDD results confirm the DataFrame results, and what do customer journeys look like? | 21, 22 |
| 8 | Can we predict whether a cart ends in a purchase? | 24 to 29 |
| 9 | What do all the results look like in charts? | 20, 30 |

## Results at a glance

| Metric | Result |
|---|---|
| Revenue | **5.13 million** from 37,343 purchases |
| Biggest driver | **Computers: 72.8% of revenue** (MSI, Gigabyte, Palit) |
| Trend | Monthly revenue 563K (Oct) to **1.38M (Feb)**; revenue per user almost tripled |
| Cart abandonment | **49.7%** |
| Best shopping hours | **09:00 to 11:00 UTC** |
| Customers | Top 29% of buyers produce about **61% of revenue** |
| Best model | Gradient Boosting, **AUC 0.67** |

**Every step has the same layout:** a heading, one code cell, then a plain-English explanation. Analysis steps end with a **Key finding** taken from the full 885K-row run.

**Two ideas to remember:** Spark is *lazy* (commands such as `filter` only build a plan, and an *action* such as `show()` or `count()` runs it) and DataFrames are *immutable* (every step returns a new DataFrame, so we save it in a variable).

**To run:** `pip install pyspark pandas pyarrow scikit-learn matplotlib jinja2` (Java 17 or newer), put `events.csv` next to the notebook (or set the `EVENTS_CSV` environment variable) and run the cells from top to bottom.

# Part 1: Setup and Loading Data

## Step 1: Import libraries

In [ ]:
# ---- Check that the extra packages are installed ----
import importlib.util

packages_needed = ["pandas", "pyarrow", "sklearn", "matplotlib", "jinja2"]
missing_packages = []
for package in packages_needed:
    if importlib.util.find_spec(package) is None:   # None means "not installed"
        missing_packages.append(package)

if missing_packages:
    print("Missing packages:", missing_packages)
    print('Run:  pip install "pyarrow>=18" pandas scikit-learn matplotlib jinja2   then restart the kernel')
else:
    print("All required packages are installed.")

# ---- Standard Python tools ----
import os                                   # work with files, folders and environment variables
import time                                 # measure how long things take
from collections import namedtuple          # small record type used in the RDD section
from operator import add                    # the "+" operator as a function (used in RDD sums)

# ---- PySpark tools ----
from pyspark.sql import SparkSession, Window, Row
from pyspark.sql import functions as F      # column functions: F.col, F.when, F.sum ...
from pyspark.sql import types as T          # data types: T.StringType, T.StructType ...
from pyspark import StorageLevel            # choose where cached data is kept
from pyspark.sql import DataFrame

# ---- Other helpers ----
import warnings
import pandas as pd                         # small tables on the driver (used to show pretty tables)
from IPython.display import display         # show tables nicely in the notebook

warnings.filterwarnings("ignore")           # hide harmless warnings to keep output tidy

**What we did and why**

The first block just checks that your computer has the extra packages. If one is missing it tells you which one and how to install it.

Then we import the tools we need. An `import` is like taking a tool out of a toolbox:

- `SparkSession` is the **front door** to Spark. You need it for DataFrames, SQL and RDDs.
- `functions as F` is a big library of column functions (`F.col`, `F.when`, `F.sum` ...). We rename it `F` so we can write `F.sum` and not mix it up with Python's own `sum`.
- `types as T` holds data types (text, number, date ...). We use them to describe the columns of a table.
- `Window` is used for "look at neighbouring rows" calculations and `Row` builds a row by hand.
- `StorageLevel` tells Spark where to keep cached data (memory, disk). `operator.add` is simply `+` written as a function.
- `pandas` and `display` help us print good-looking tables. `warnings.filterwarnings("ignore")` hides warning messages.

## Step 2: Create the SparkSession and SparkContext

In [ ]:
import sys

# Make Spark's Python workers use the SAME Python as this notebook.
# This fixes the common Windows error "Python worker failed to connect back".
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("EcommerceClickstreamAnalysis")         # a name shown in the Spark web page
    .master("local[*]")                              # run on this computer, use every CPU core
    .config("spark.sql.shuffle.partitions", "8")     # default is 200, far too many for local data
    .config("spark.driver.memory", "4g")             # give Spark 4 GB of memory
    .config("spark.sql.session.timeZone", "UTC")     # keep all time handling in UTC
    .config("spark.python.worker.faulthandler.enabled", "true")  # show the real error if a worker crashes
    .config("spark.ui.showConsoleProgress", "false") # no progress bars in the output
    .getOrCreate()                                   # reuse a session if one already exists
)

sc = spark.sparkContext      # the low-level "RDD" entry point (used in Part 12)
sc.setLogLevel("ERROR")      # hide routine log lines, show only real errors

print("Spark version      :", spark.version)
print("Master             :", sc.master)
print("Default parallelism:", sc.defaultParallelism)

**What we did and why**

`SparkSession.builder` is a **settings form**. Each `.config(...)` line fills in one setting, and `.getOrCreate()` submits the form and starts Spark.

- `master("local[*]")` means "run on my own computer and use all CPU cores". On a real cluster this would be a cluster address instead.
- `spark.sql.shuffle.partitions` is how many pieces Spark splits data into after a *shuffle* (a join or group-by). The default of 200 is for giant clusters. For a laptop, 8 is much faster.
- `spark.sparkContext` (we call it `sc`) is the door to the RDD API used in Part 12.
- `setLogLevel("ERROR")` hides routine messages so you only see real problems.
- The two `os.environ[...]` lines make Spark use the same Python as your notebook. If they differ, RDD code can fail with strange "worker" errors.
- **Restart the kernel** if you change any of these settings, because they are only read once, when the session is created.
- While Spark runs, you can open http://localhost:4040 to see a live web page of jobs and stages.

## Step 3: Professional table output for every DataFrame

In [ ]:
# ------------------------------------------------------------------
# TABLE STYLE (kept exactly as in the original project)
# This cell makes every df.show() print a nice styled table.
# ------------------------------------------------------------------
_TABLE_STYLES = [
    {"selector": "th", "props": [("background-color", "#1f4e79"), ("color", "white"), ("text-align", "left"),
                                 ("padding", "6px 14px"), ("font-weight", "600")]},
    {"selector": "td", "props": [("padding", "5px 14px"), ("border-bottom", "1px solid rgba(128,128,128,0.25)")]},
    {"selector": "tbody tr:nth-child(even)", "props": [("background-color", "rgba(128,128,128,0.10)")]},
    {"selector": "caption", "props": [("caption-side", "top"), ("font-weight", "600"),
                                      ("text-align", "left"), ("padding", "4px 0")]},
]

def show_pdf(pdf, title=None, highlight_max=None):
    """Show a pandas DataFrame as a styled HTML table."""
    # show decimals with a thousands separator and 2 digits, e.g. 1,234.50
    fmt = {c: "{:,.2f}" for c in pdf.columns if pd.api.types.is_float_dtype(pdf[c])}
    styler = pdf.style.hide(axis="index").format(fmt, na_rep="-").set_table_styles(_TABLE_STYLES)
    if highlight_max:
        # colour the best (largest) value of each listed column green
        styler = styler.highlight_max(subset=highlight_max, color="#c6efce", props="color:#006100;font-weight:bold;")
    if title:
        styler = styler.set_caption(title)
    display(styler)

_original_show = DataFrame.show      # remember Spark's own show() in case ours fails

def pretty_show(self, n=20, truncate=True, vertical=False, title=None):
    """Drop-in replacement for DataFrame.show() that prints a styled table."""
    try:
        pdf = self.limit(int(n)).toPandas()          # bring only n rows to pandas
        width = 40 if truncate is True else (int(truncate) if truncate else None)
        if width:
            # shorten very long text so the table stays readable
            for c in pdf.columns:
                if pdf[c].dtype == object:
                    pdf[c] = pdf[c].map(lambda v: v if not isinstance(v, str) or len(v) <= width else v[:width - 3] + "...")
        if vertical:
            # one row shown as "field / value" pairs
            pdf = pdf.head(1).T.reset_index()
            pdf.columns = ["field", "value"]
            pdf["value"] = pdf["value"].map(lambda v: f"{v:,.2f}" if isinstance(v, float) else str(v))
        show_pdf(pdf, title)
    except Exception:
        _original_show(self, n, truncate, vertical)  # fall back to the plain Spark table

# In Spark 4 the DataFrame you actually get is a subclass that overrides show(), so patch that class too
for _cls in {DataFrame, type(spark.range(1))}:
    _cls.show = pretty_show           # every df.show(...) below now renders as a styled table

# Small test so you can see the new table style
clean_demo = spark.range(3).select((F.col("id") + 1).alias("rank"), (F.col("id") * 1234.5).alias("amount"))
clean_demo.show(title="Preview of the new table style")

**What we did and why**

Spark's normal `df.show()` prints plain text. This cell replaces it with a good-looking table (dark blue header, striped rows, thousands separators). **You do not need to understand every line of this cell. Just run it once.** Every later `df.show()` then looks nice automatically.

What happens inside:

- `_TABLE_STYLES` is a list of CSS style rules (colours, padding) for the table header, cells, stripes and title. The colours are semi-transparent, so they look right in both light and dark themes.
- `show_pdf(...)` takes a small **pandas** table and displays it with those styles. Decimal numbers get two digits. It can also colour the best value of a column green (`highlight_max`). We reuse it later for model results.
- `pretty_show(...)` is our new version of `show()`. It takes only the first `n` rows (`limit(n)`), converts them with `toPandas()`, shortens long text, and calls `show_pdf`. If anything goes wrong it falls back to Spark's original `show()`.
- The last `for` loop swaps Spark's `show` for ours. Replacing a function like this is called *monkey-patching*.
- Only a few rows are ever brought to your computer, so this is safe for big data.

## Step 4: Define an explicit schema and read the CSV

In [ ]:
# Where is the file? Use the EVENTS_CSV environment variable if set, otherwise "events.csv" next to the notebook
DATA_PATH = os.path.abspath(os.environ.get("EVENTS_CSV", "events.csv"))

# A schema is the list of column names and their data types
schema = T.StructType([
    T.StructField("event_time",    T.StringType(),  True),   # True = this column may contain empty values
    T.StructField("event_type",    T.StringType(),  True),
    T.StructField("product_id",    T.LongType(),    True),   # Long = big whole number
    T.StructField("category_id",   T.LongType(),    True),
    T.StructField("category_code", T.StringType(),  True),
    T.StructField("brand",         T.StringType(),  True),
    T.StructField("price",         T.DoubleType(),  True),   # Double = number with decimals
    T.StructField("user_id",       T.LongType(),    True),
    T.StructField("user_session",  T.StringType(),  True),
])

raw_df = (
    spark.read
    .option("header", True)            # the first line holds the column names
    .option("mode", "PERMISSIVE")      # keep bad rows instead of failing
    .schema(schema)                    # use our schema instead of guessing
    .csv(DATA_PATH)                    # read the CSV file
)

raw_df.show(5, truncate=False)         # show the first 5 rows

**What we did and why**

- A **schema** tells Spark the name and type of every column. A `StructType` is the whole list and each `StructField(name, type, nullable)` describes one column. Writing the schema ourselves is faster than `inferSchema=True` (Spark would need to scan the file once just to guess) and safer (the long ID numbers must be `LongType`).
- `spark.read` is the **file reader**. `option("header", True)` means "first line = column names". `mode="PERMISSIVE"` means "keep broken rows" (the other choices are `DROPMALFORMED` = throw them away, and `FAILFAST` = stop with an error).
- Reading is **lazy**: Spark only notes *"I will read this file"*. `show(5)` is an **action**, so it triggers the real work.
- `event_time` stays text for now because values like `2020-09-24 11:57:06 UTC` must be cleaned before they can become real timestamps.

## Step 5: Inspect the data: schema, size, basic statistics

In [ ]:
raw_df.printSchema()                                  # column names, types, nullable
print("Rows   :", raw_df.count())                     # number of rows (scans the whole file)
print("Columns:", len(raw_df.columns), raw_df.columns)
print("dtypes :", raw_df.dtypes)                      # list of (column, type)

raw_df.describe("price").show()                       # count, mean, stddev, min, max
raw_df.select("price").summary("min", "25%", "50%", "75%", "99%", "max").show()   # percentiles

**What we did and why**

- `printSchema()` prints the columns as a small tree: name, type and whether empty values are allowed.
- `count()` counts the rows. It is an **action**, so it reads the file. `columns` and `dtypes` are cheap because they only read the schema.
- `describe()` gives a quick summary (count, average, spread, smallest, largest). `summary()` is similar but lets you pick percentiles such as `25%`, `50%` (the median) and `99%`.
- If the 99% value is much smaller than the maximum, a few very expensive items are pulling the average up. Remember this when we study revenue later.

> **Key finding:** The price distribution is heavily skewed. The median price is 65.71 but the mean is 146.33, the 99th percentile is 886.67 and the maximum is 64,771. A small number of very expensive items pull averages up, so medians and percentiles are used alongside averages in the analysis.

# Part 2: Data Cleaning and Feature Engineering

## Step 6: Convert the timestamp string to a real timestamp

In [ ]:
# 1) remove the trailing " UTC" from the text, 2) turn the text into a real timestamp
event_time_without_utc = F.regexp_replace("event_time", r"\s*UTC$", "")
event_timestamp = F.to_timestamp(event_time_without_utc, "yyyy-MM-dd HH:mm:ss")

df = (
    raw_df
    .withColumn("event_ts", event_timestamp)   # add the new timestamp column
    .drop("event_time")                        # remove the old text column
)

df.select("event_ts", "event_type", "price").show(3, truncate=False)
df.printSchema()

**What we did and why**

- `F.regexp_replace(column, pattern, replacement)` finds text that matches a pattern and replaces it. Here the pattern `\s*UTC$` means "optional spaces, then `UTC`, at the very end", and we replace it with nothing.
- `F.to_timestamp(text, format)` reads the cleaned text using the format `yyyy-MM-dd HH:mm:ss` and gives a real **timestamp**. Real timestamps unlock hour, day, week and date-difference functions.
- `withColumn(name, expression)` adds a column (or replaces one with the same name). Remember: a DataFrame never changes, so we save the result in a new variable `df`.
- `drop()` removes the old text column.

## Step 7: Profile missing values

In [ ]:
# Count missing (NULL) values in every column, in a single pass over the data.
# when(isNull) gives a value only for missing cells, and count() counts those values.
null_count_expressions = []
for column_name in df.columns:
    missing_only = F.when(F.col(column_name).isNull(), column_name)
    null_count_expressions.append(F.count(missing_only).alias(column_name))

null_counts = df.select(null_count_expressions)
null_counts.show(vertical=True)

# Percentage version for three important columns
total = df.count()
percent_expressions = []
for column_name in ["category_code", "brand", "user_session"]:
    is_missing_as_number = F.col(column_name).isNull().cast("int")      # 1 if missing, else 0
    percent_missing = F.round(F.sum(is_missing_as_number) / total * 100, 2)
    percent_expressions.append(percent_missing.alias(column_name))

null_pct = df.select(percent_expressions)
null_pct.show()

**What we did and why**

- A missing value in Spark is called `NULL`. `isNull()` is true when a cell is empty.
- `F.when(condition, value)` returns `value` when the condition is true, otherwise `NULL`. `F.count` skips `NULL`s, so `count(when(isNull))` counts exactly the missing cells.
- The `for` loop builds one expression per column, so Spark checks **all columns in one pass**.
- For the percentage version we turn "missing" into `1` and "present" into `0` with `cast("int")`, add them up with `F.sum`, divide by the total rows and multiply by 100.
- Result: `category_code` and `brand` are missing in about a quarter of the rows (common for less popular products), while `user_session` is almost always present.

> **Key finding:** Data quality is good overall. Only 165 rows (0.02%) have no session id, while `category_code` (26.69%) and `brand` (23.99%) are missing for roughly a quarter of the events. These gaps are labelled `unknown` instead of being dropped, so no sales are lost.

## Step 8: Clean the data: drop, filter, deduplicate, fill

In [ ]:
clean = (
    df
    .dropna(subset=["user_session"])                    # sessions are essential, so drop rows without one
    .filter(F.col("price") > 0)                         # remove zero or invalid prices
    .dropDuplicates()                                   # remove rows that are exact copies
    .fillna({"brand": "unknown", "category_code": "unknown"})   # replace missing values with "unknown"
)

print("Before cleaning:", df.count())
print("After cleaning :", clean.count())

**What we did and why**

Cleaning is done as a chain of four small steps:

- `dropna(subset=[...])` deletes rows where the listed column is empty.
- `filter(condition)` keeps only the rows that match (here: price above zero). `where()` is the same thing.
- `dropDuplicates()` deletes rows that are completely identical. You can pass a list of columns to compare only those, for example `["user_session", "event_ts", "product_id"]`.
- `fillna({...})` replaces empty values. Missing brands and categories become `"unknown"`, so they appear as their own group in later group-bys instead of silently vanishing.

Printing the row count **before and after** is a good habit: it shows how much data the cleaning removed.

> **Key finding:** Cleaning removed only 817 of 885,129 rows (0.09%), leaving **884,312 clean events**. The dataset is reliable for analysis.

## Step 9: Date and time features

In [ ]:
clean = (
    clean
    .withColumn("event_date", F.to_date("event_ts"))                # only the date part
    .withColumn("year_month", F.date_format("event_ts", "yyyy-MM")) # e.g. 2020-10
    .withColumn("hour",       F.hour("event_ts"))                   # hour of the day, 0 to 23
    .withColumn("dow",        F.dayofweek("event_ts"))              # day of week: 1 = Sunday ... 7 = Saturday
    .withColumn("day_name",   F.date_format("event_ts", "E"))       # Mon, Tue, ...
    .withColumn("is_weekend", F.col("dow").isin(1, 7))              # True on Saturday and Sunday
    .withColumn("week_start", F.to_date(F.date_trunc("week", "event_ts")))  # Monday of that week
)

clean.select("event_ts", "event_date", "year_month", "hour", "day_name", "is_weekend", "week_start").show(5)

# A few more date helpers on a tiny example (one row, no real data needed)
spark.range(1).select(
    F.current_date().alias("today"),
    F.date_add(F.current_date(), 30).alias("in_30_days"),
    F.datediff(F.lit("2021-02-28").cast("date"), F.lit("2020-09-24").cast("date")).alias("dataset_span_days"),
    F.months_between(F.lit("2021-02-28").cast("date"), F.lit("2020-09-24").cast("date")).alias("months"),
).show()

**What we did and why**

- From one timestamp we create many handy columns: the date, the month, the hour, the day of the week, a weekend flag and the start of the week. We will use them later as **group-by keys** ("revenue per month", "purchases per hour").
- `isin(1, 7)` is true when the value is 1 or 7, so `is_weekend` is `True` for Sunday (1) and Saturday (7).
- The small `spark.range(1)` example creates a one-row table so we can try date functions on their own: `current_date` (today), `date_add` (add days), `datediff` (days between two dates) and `months_between`.
- More date functions you may meet: `year`, `month`, `dayofmonth`, `weekofyear`, `last_day`, `add_months`, `unix_timestamp`, `from_unixtime`.

## Step 10: Category, price and revenue features

In [ ]:
# Split the dotted category (computers.components.cooler) into a list of words
parts = F.split("category_code", r"\.")

clean = (
    clean
    .withColumn("main_category", F.element_at(parts, 1))      # 1st word, e.g. computers
    .withColumn("sub_category",                               # 2nd word, or "unknown" if there is none
                F.coalesce(F.when(F.size(parts) > 1, F.element_at(parts, 2)), F.lit("unknown")))
    .withColumn("leaf_category", F.element_at(parts, -1))     # last word, e.g. cooler
    .withColumn("brand_upper", F.upper(F.trim("brand")))      # APPLE (spaces removed)
    .withColumn("brand_title", F.initcap("brand"))            # Apple
    .withColumn("brand_len",   F.length("brand"))             # number of letters
    .withColumn("category_path", F.concat_ws(" > ", "main_category", "sub_category"))   # computers > components
    .withColumn(
        "price_tier",                                         # label each price: budget / mid / high / premium
        F.when(F.col("price") < 20, "budget")
         .when(F.col("price") < 100, "mid")
         .when(F.col("price") < 500, "high")
         .otherwise("premium")
    )
    .withColumn("revenue",                                    # money earned: price only for purchases
                F.when(F.col("event_type") == "purchase", F.col("price")).otherwise(F.lit(0.0)))
    .withColumn("is_known_brand", F.col("brand") != "unknown")
)

clean.select("category_code", "main_category", "sub_category", "leaf_category",
             "brand_title", "category_path", "price", "price_tier").show(8, truncate=False)

# Keep the cleaned data in memory, because almost every step below reuses it
clean.cache()
print("Cached rows:", clean.count())

**What we did and why**

- `F.split(column, pattern)` cuts text into a list at every dot. `F.element_at(list, 1)` takes the **first** item (counting starts at 1), and `-1` takes the **last** item. `F.size(list)` is the length of the list.
- `F.coalesce(a, b)` returns the first value that is not empty. `F.lit("unknown")` turns a plain Python value into a column.
- String helpers: `upper` (CAPITALS), `trim` (remove spaces at the ends), `initcap` (Capitalise Each Word), `length` and `concat_ws` (join texts with a separator). Others to explore: `substring`, `lpad`, `regexp_extract`, `translate`.
- `F.when(...).when(...).otherwise(...)` is Spark's **if / elif / else**. It builds `price_tier` and the `revenue` column.
- `cache()` says "keep this table in memory the first time it is computed". The `count()` right after forces that to happen. After this, later steps read from memory and do not re-read the CSV.

# Part 3: Revenue Analysis

## Step 11: Business KPIs

In [ ]:
kpis = clean.agg(
    F.count("*").alias("total_events"),                              # number of rows
    F.countDistinct("user_id").alias("unique_users"),                # exact unique count
    F.countDistinct("user_session").alias("unique_sessions"),
    F.countDistinct("product_id").alias("unique_products"),
    F.approx_count_distinct("brand").alias("approx_brands"),         # fast, approximate unique count
    F.min("event_ts").alias("first_event"),
    F.max("event_ts").alias("last_event"),
    F.round(F.sum("revenue"), 2).alias("total_revenue"),
    F.round(F.avg("price"), 2).alias("avg_price"),
    F.round(F.stddev("price"), 2).alias("std_price"),                # how spread out prices are
    F.expr("percentile_approx(price, 0.5)").alias("median_price"),   # the middle price
)
kpis.show(vertical=True, truncate=False)

**What we did and why**

- `agg(...)` without `groupBy` squeezes the **whole table into one row** of totals (our KPIs: key numbers of the business).
- `countDistinct` counts unique values exactly but is slow on big data. `approx_count_distinct` is a fast estimate (using an algorithm called HyperLogLog).
- `min`, `max`, `sum`, `avg` and `stddev` are the standard summary functions. The median uses `percentile_approx`, written through `F.expr`.
- `vertical=True` prints one field per line, which is easier to read when there are many columns.

> **Key finding:** The store recorded **5.13 million** in purchase revenue from 407,237 unique customers across 490,398 sessions and 53,452 products. The typical (median) item price is 65.71.

## Step 12: Revenue by main category

In [ ]:
# How many events of each type, and what are their prices like?
clean.groupBy("event_type").agg(
    F.count("*").alias("events"),
    F.round(F.avg("price"), 2).alias("avg_price"),
    F.round(F.max("price"), 2).alias("max_price"),
).orderBy(F.desc("events")).show()

# Revenue by main category (purchases only)
purchases_only = clean.filter("event_type = 'purchase'")        # filter FIRST so less data is grouped

rev_by_category = (
    purchases_only
    .groupBy("main_category")
    .agg(
        F.count("*").alias("items_sold"),
        F.round(F.sum("price"), 2).alias("revenue"),
        F.round(F.avg("price"), 2).alias("avg_item_price"),
        F.countDistinct("user_id").alias("buyers"),
    )
    .orderBy(F.desc("revenue"))
)
rev_by_category.show(10)

**What we did and why**

- `groupBy(column).agg(...)` is like a pivot-table summary: Spark makes one group per value and calculates several results for each group in a single pass.
- **Always give each result an `alias`**. Otherwise Spark names it something ugly like `sum(price)`.
- Shortcuts exist for a single result: `.count()`, `.sum("col")`, `.avg("col")`.
- We filter to purchases **before** grouping. Filtering early means fewer rows to shuffle, which is a general speed rule in Spark.
- Sorting by revenue shows which categories earn the most.

> **Key finding:** Browsing dominates: 792,943 views against 54,026 cart events and 37,343 purchases. **Computers generate 3.73 million, or 72.8% of all revenue**, with an average item price of 221. Electronics sells many items at a much lower price (66.54).

## Step 13: Top brands and top products

In [ ]:
top_brands = (
    clean.filter("event_type = 'purchase' AND brand != 'unknown'")   # ignore the "unknown" bucket
    .groupBy("brand")
    .agg(F.count("*").alias("purchases"), F.round(F.sum("price"), 2).alias("revenue"))
    .orderBy(F.desc("revenue"))
)
top_brands.show(10)

top_products = (
    clean.filter("event_type = 'purchase'")
    .groupBy("product_id", "leaf_category")                          # one group per product + category pair
    .agg(F.count("*").alias("purchases"))
    .orderBy(F.desc("purchases"))
    .limit(10)                                                       # keep the top 10
)
top_products.show()

**What we did and why**

- Grouping by **two columns** (`product_id`, `leaf_category`) creates one group for every unique pair.
- `orderBy(...).limit(10)` is a "top 10" query. Spark is smart enough not to sort the whole table to answer it.
- We skip brand `unknown` because it just means "data missing", not a real brand.

> **Key finding:** Revenue is concentrated in PC components. The top brands are **MSI (643K), Gigabyte (556K) and Palit (484K)**, and the best-selling products are mostly video cards. Product 4099645 alone had 564 purchases.

## Step 14: Product conversion: combining tables with joins

In [ ]:
# Table 1: how many views each product got
views_by_product = (
    clean.filter("event_type = 'view'").groupBy("product_id").agg(F.count("*").alias("views"))
)

# Table 2: how many purchases and how much revenue each product made
buys_by_product = (
    clean.filter("event_type = 'purchase'")
    .groupBy("product_id").agg(F.count("*").alias("purchases"), F.round(F.sum("price"), 2).alias("revenue"))
)

# Table 3: a small "description" table (brand, category, average price) for each product.
# max() ignores NULLs, so max(when(brand is known)) returns the known brand if one exists.
product_dim = (
    clean.groupBy("product_id")
    .agg(
        F.max(F.when(F.col("brand") != "unknown", F.col("brand"))).alias("brand"),
        F.max(F.when(F.col("main_category") != "unknown", F.col("main_category"))).alias("main_category"),
        F.round(F.avg("price"), 2).alias("avg_price"),
    )
    .fillna("unknown")
)

# Join the three tables together
product_stats = (
    views_by_product
    .join(buys_by_product, on="product_id", how="left")              # keep every viewed product
    .join(F.broadcast(product_dim), on="product_id", how="left")     # broadcast = send the small table everywhere
    .fillna({"purchases": 0, "revenue": 0.0})                        # never bought -> 0
    .withColumn("conversion_pct", F.round(F.col("purchases") / F.col("views") * 100, 2))   # purchases per 100 views
)

# Only products with at least 200 views (small numbers give misleading percentages)
product_stats.filter("views >= 200").orderBy(F.desc("conversion_pct")).show(10)

**What we did and why**

- We build three small summary tables at product level and combine them with `join(other, on=key, how=type)`.
- A **left join** keeps every row of the left table (all viewed products) even when there is no match. The empty cells it creates are replaced with 0 using `fillna`.
- `F.broadcast(small_table)` is a hint: "this table is small, copy it to every worker". That avoids shuffling the big table, which makes the join faster.
- The `max(when(...))` trick finds "the first real value": `max` ignores `NULL`s, so unknown brands are skipped.
- `views >= 200` hides products with very few views, because their conversion percentage would be meaningless.

> **Key finding:** Conversion is highly product-specific. Among products with at least 200 views, the best converters reach **20% to 28%** of views ending in a purchase, and they are mostly cheap products (average price 1.57 to 27). These are natural candidates for cross-selling and bundles. Only **6,434 of 53,452 viewed products (12%) were ever purchased**. The other 47,017 were viewed but never bought.

# Part 4: Trends and Customer Behaviour

## Step 15: Monthly trend: events per month by type (pivot)

In [ ]:
monthly = (
    clean.groupBy("year_month")
    .pivot("event_type", ["view", "cart", "purchase"])      # turn each event type into its own column
    .count()
    .orderBy("year_month")
    .withColumn("cart_rate_pct",     F.round(F.col("cart") / F.col("view") * 100, 2))       # carts per 100 views
    .withColumn("purchase_rate_pct", F.round(F.col("purchase") / F.col("view") * 100, 2))   # purchases per 100 views
)
monthly.show()

**What we did and why**

- `pivot` turns the **values of one column into separate columns**. The three event types become three columns, giving a compact cross-table.
- We list the values (`["view", "cart", "purchase"]`) ourselves. This saves Spark one extra job that would otherwise look up the values first.
- After pivoting you can still add calculated columns, such as the cart rate and purchase rate.
- The first and last months are only partly covered by the data (it starts on 24 Sept and ends on 28 Feb), so compare **full months** only.

> **Key finding:** The business grew steadily. The cart rate rose from 5.5% to 7.7% of views and the purchase rate from 3.9% to 5.2% between September 2020 and February 2021. September is a partial month (the data starts on 24 September).

## Step 16: Daily revenue: running total and 7-day average (window functions)

In [ ]:
# Revenue and number of orders per day
daily = (
    clean.filter("event_type = 'purchase'")
    .groupBy("event_date")
    .agg(F.round(F.sum("price"), 2).alias("revenue"), F.count("*").alias("orders"))
)

w_all = Window.orderBy("event_date")                                   # all days in date order
w_run = w_all.rowsBetween(Window.unboundedPreceding, Window.currentRow)  # from the first day up to today
w_7d  = w_all.rowsBetween(-6, 0)                                       # the last 7 days including today

daily = (
    daily
    .withColumn("cumulative_revenue", F.round(F.sum("revenue").over(w_run), 2))     # running total
    .withColumn("moving_avg_7d",      F.round(F.avg("revenue").over(w_7d), 2))      # 7-day average
    .withColumn("pct_of_prev_day",                                                  # today as % of yesterday
                F.round(F.col("revenue") / F.lag("revenue", 1).over(w_all) * 100, 1))
    .orderBy("event_date")
)

daily.show(10)
daily.orderBy(F.desc("revenue")).show(5)       # the best revenue days

**What we did and why**

- `rowsBetween(start, end)` chooses **which neighbouring rows** the window can see, relative to the current row.
  - `(unboundedPreceding, currentRow)` = everything from the start up to now, giving a **running total**.
  - `(-6, 0)` = the 6 rows before plus the current one, giving a **7-day moving average**.
- `F.lag("revenue", 1)` reads the value from 1 row earlier, so we can compare each day with the day before.
- A window with **no** `partitionBy` puts all rows into a single group. That is fine here (only ~160 days), but it becomes slow on big data.
- The second table lists the best revenue days. They often match promotions such as Black Friday.

> **Key finding:** Daily revenue grew from roughly 8,000 to 25,000 in the last days of September 2020 to **60,000 to 65,000 per day in January and February 2021**. The best day was 24 January 2021 (65,588 from 294 orders).

## Step 17: Conversion funnel and cart abandonment

In [ ]:
# One row per session with 0/1 flags: did the session contain a view / cart / purchase?
sessions = clean.groupBy("user_session").agg(
    F.max((F.col("event_type") == "view").cast("int")).alias("viewed"),
    F.max((F.col("event_type") == "cart").cast("int")).alias("carted"),
    F.max((F.col("event_type") == "purchase").cast("int")).alias("purchased"),
    F.sum("revenue").alias("session_revenue"),
)

# Count sessions at each stage of the funnel
funnel = sessions.agg(
    F.count("*").alias("sessions"),
    F.sum("viewed").alias("sessions_with_view"),
    F.sum("carted").alias("sessions_with_cart"),
    F.sum("purchased").alias("sessions_with_purchase"),
    F.sum(F.col("viewed") * F.col("carted")).alias("view_and_cart"),            # both flags = 1
    F.sum(F.col("carted") * F.col("purchased")).alias("cart_and_purchase"),
).first()                                                                        # .first() gives the single result row

print(f"Sessions                : {funnel['sessions']:,}")
print(f"Sessions with a view    : {funnel['sessions_with_view']:,}")
print(f"Sessions with a cart    : {funnel['sessions_with_cart']:,}")
print(f"Sessions with a purchase: {funnel['sessions_with_purchase']:,}")
print(f"View -> cart rate       : {funnel['view_and_cart'] / funnel['sessions_with_view']:.2%}")
print(f"Cart -> purchase rate   : {funnel['cart_and_purchase'] / funnel['sessions_with_cart']:.2%}")
print(f"Cart abandonment rate   : {1 - funnel['cart_and_purchase'] / funnel['sessions_with_cart']:.2%}")

**What we did and why**

- A **funnel** shows how many users make it from one stage to the next: view, then cart, then purchase.
- We turn each session into three flags (1 = yes, 0 = no). `max(condition cast to int)` is a common pattern for "did this happen **at least once**?".
- One `agg` then counts sessions per stage. **Multiplying two flags** (`viewed * carted`) counts sessions where both happened.
- `.first()` returns the single result row to Python, and fields are read like a dictionary: `funnel['sessions']`.
- **Cart abandonment** = share of cart sessions that never reach a purchase. A purchase can appear in a session that has no cart or view event, which is why each rate is based on the sessions that reached the earlier stage.

> **Key finding:** Out of 490,398 sessions, 41,270 contain a cart (a **view-to-cart rate of 8.45%**) and 24,344 contain a purchase. Half of the cart sessions convert (**50.31%**), so the **cart abandonment rate is 49.69%**. Fixing abandonment is the biggest short-term revenue opportunity.

## Step 18: Shopping hours and weekdays

In [ ]:
# Events per hour of the day, split by event type
hourly = (
    clean.groupBy("hour").pivot("event_type", ["view", "cart", "purchase"]).count()
    .withColumn("purchase_per_1000_views", F.round(F.col("purchase") / F.col("view") * 1000, 1))
    .orderBy("hour")
)
hourly.show(24)

# Events and revenue per weekday (dow keeps calendar order, day_name is readable)
(clean.groupBy("dow", "day_name")
      .agg(F.count("*").alias("events"), F.round(F.sum("revenue"), 0).alias("revenue"))
      .orderBy("dow").show())

**What we did and why**

- Grouping by `hour` and pivoting by event type shows **when people browse and when they buy**.
- Grouping by both `dow` (a number) and `day_name` (text) keeps the days in calendar order but still shows readable names.
- Use these results to choose when to run campaigns or maintenance. Remember: the hours are in **UTC**, not the shop's local time.

> **Key finding:** Shopping follows the working day. Views peak at 11:00 UTC (47,575) and purchases at 10:00 UTC (2,535). Conversion is best between 09:00 and 11:00 UTC (about 53 purchases per 1,000 views) and lowest at night (36.1 at 01:00 UTC). Monday earns the most revenue (771,508) and Saturday the least (674,545); weekends are 27.4% of revenue.

## Step 19: Customer segmentation with RFM

In [ ]:
buyers = clean.filter("event_type = 'purchase'")
snapshot_date = buyers.agg(F.max("event_date")).first()[0]     # latest date in the data acts as "today"

# For every buyer: Recency, Frequency, Monetary
rfm = buyers.groupBy("user_id").agg(
    F.datediff(F.lit(snapshot_date), F.max("event_date")).alias("recency_days"),   # days since last purchase
    F.countDistinct("user_session").alias("frequency"),                            # number of purchase sessions
    F.round(F.sum("price"), 2).alias("monetary"),                                  # total money spent
)

rfm = (
    rfm
    # ntile(4) cuts customers into 4 equal groups (quartiles). Score 4 is the best.
    .withColumn("r_score", F.ntile(4).over(Window.orderBy(F.desc("recency_days"))))   # 4 = most recent
    .withColumn("m_score", F.ntile(4).over(Window.orderBy("monetary")))               # 4 = biggest spenders
    # frequency is mostly 1, so we score it by hand
    .withColumn("f_score", F.when(F.col("frequency") >= 3, 3).when(F.col("frequency") == 2, 2).otherwise(1))
    # turn the scores into business-friendly segment names
    .withColumn(
        "segment",
        F.when((F.col("r_score") >= 3) & (F.col("m_score") >= 3) & (F.col("f_score") >= 2), "Champions")
         .when((F.col("r_score") >= 3) & (F.col("m_score") >= 3), "Big new spenders")
         .when(F.col("r_score") <= 2, "At risk / lapsed")
         .otherwise("Regular")
    )
)

rfm.groupBy("segment").agg(
    F.count("*").alias("customers"),
    F.round(F.avg("monetary"), 2).alias("avg_spend"),
    F.round(F.avg("recency_days"), 1).alias("avg_recency"),
).orderBy(F.desc("customers")).show()

**What we did and why**

**RFM** scores each customer on three things:

- **R**ecency: how many days since the last purchase (fewer is better).
- **F**requency: how many purchase sessions.
- **M**onetary: how much money in total.

Other points:

- `F.datediff` compares each customer's last purchase with the **latest date in the data**, which stands in for "today". This keeps the analysis valid for an old dataset.
- `F.ntile(4)` splits customers into four equal groups. Sorting the window ascending or descending decides which end gets score 4.
- Frequency is mostly 1 here, so it is scored by hand with `when`.
- The final `when` chain turns numbers into names a marketing team can act on (Champions, At risk, ...).

> **Key finding:** Customers fall into four segments: **Champions** (1,141 customers, average spend 844), **Big new spenders** (5,049 customers, 425), **Regular** (4,462 customers, 39) and **At risk / lapsed** (10,652 customers, 173, last purchase about 113 days ago). Champions and Big new spenders are 29% of buyers but generate about 61% of revenue, while the lapsed segment is the largest and represents roughly 36% of revenue that is at risk.

## Step 20: Dashboard of the key results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# A clean chart style
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 12,
                     "axes.spines.top": False, "axes.spines.right": False})

# Bring the SMALL summary tables to pandas (never plot raw big data)
cat_pdf   = rev_by_category.limit(8).toPandas().sort_values("revenue")
daily_pdf = daily.orderBy("event_date").toPandas()
daily_pdf["event_date"] = pd.to_datetime(daily_pdf["event_date"])
hour_pdf  = hourly.toPandas().sort_values("hour")

# Show axis numbers as thousands, e.g. 12K
k_fmt = mtick.FuncFormatter(lambda x, _: f"{x/1e3:,.0f}K")

fig, ax = plt.subplots(2, 2, figsize=(14, 9))      # a 2 x 2 grid of charts

# Top-left: revenue by main category
ax[0, 0].barh(cat_pdf["main_category"], cat_pdf["revenue"], color="#1f4e79")
ax[0, 0].xaxis.set_major_formatter(k_fmt)
ax[0, 0].set_title("Revenue by main category (top 8)")

# Top-right: daily revenue and its 7-day average
ax[0, 1].plot(daily_pdf["event_date"], daily_pdf["revenue"], color="#85c1e9", label="Daily revenue")
ax[0, 1].plot(daily_pdf["event_date"], daily_pdf["moving_avg_7d"], color="#1f4e79", linewidth=2, label="7-day average")
ax[0, 1].yaxis.set_major_formatter(k_fmt)
ax[0, 1].set_title("Daily purchase revenue")
ax[0, 1].legend(frameon=False)

# Bottom-left: purchases by hour
ax[1, 0].bar(hour_pdf["hour"], hour_pdf["purchase"], color="#e67e22")
ax[1, 0].set_xticks(range(0, 24, 2))
ax[1, 0].set_title("Purchases by hour of day (UTC)")

# Bottom-right: the session funnel
stages = ["Sessions with view", "Sessions with cart", "Sessions with purchase"]
values = [funnel["sessions_with_view"], funnel["sessions_with_cart"], funnel["sessions_with_purchase"]]
bars = ax[1, 1].bar(stages, values, color=["#2e86c1", "#f39c12", "#27ae60"])
ax[1, 1].yaxis.set_major_formatter(k_fmt)
ax[1, 1].set_title("Session funnel")
for b, v in zip(bars, values):                      # write the number above each bar
    ax[1, 1].text(b.get_x() + b.get_width() / 2, v, f"{v:,}", ha="center", va="bottom", fontsize=9)

fig.suptitle("E-commerce Clickstream: Key Results", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

**What we did and why**

- The dashboard reuses tables we already built (`rev_by_category`, `daily`, `hourly`, `funnel`). We convert only these **small summaries** to pandas with `toPandas()`. Rule of thumb: **summarise in Spark first, then plot the small result**.
- The four charts: top-left revenue by category, top-right daily revenue with its 7-day average (from the window step), bottom-left purchases by hour, bottom-right the funnel.
- `plt.subplots(2, 2)` creates a 2 by 2 grid. `ax[row, column]` picks one chart in the grid.
- `FuncFormatter` shows big numbers as `K` (thousands). The style and `rcParams` give a clean look with the top and right borders removed.

> **Key finding:** The dashboard summarises the story in four charts: computers dominate revenue, daily revenue climbs through the season, buying peaks in the morning (UTC) and the funnel narrows sharply from view to cart to purchase.

# Part 5: Low-Level Processing with the RDD API

**Goal:** recompute key numbers with Spark's lower-level **RDD** API. This (a) double-checks the DataFrame results and (b) shows what RDDs add: custom parsing, shared variables (accumulator, broadcast variable) and full control over how records are grouped. An RDD is simply a distributed collection of Python objects with no schema. For table-like data DataFrames are easier, so the main analysis uses DataFrames and this part is the cross-check.

## Step 21: RDD part 1: parse the raw file, count events and revenue by brand

In [ ]:
Event = namedtuple("Event", ["event_time", "event_type", "product_id", "category_code",
                             "brand", "price", "user_id", "session"])
bad_rows = sc.accumulator(0)          # shared counter: workers add, the driver reads

# Read the raw file as text lines (fall back to the DataFrame reader if textFile fails on this machine)
raw_rdd = sc.textFile(DATA_PATH, minPartitions=8)
try:
    raw_rdd.first()
except Exception:
    def first_column(row):
        return row[0]
    raw_rdd = spark.read.text(DATA_PATH).rdd.map(first_column)

def parse_partition(lines):
    """Turn the text lines of ONE partition into Event records."""
    import csv
    for row in csv.reader(lines):
        if row[0] == "event_time":                       # skip the header line
            continue
        try:
            yield Event(row[0], row[1], int(row[2]), row[4] or "unknown", row[5] or "unknown",
                        float(row[6]), int(row[7]), row[8])
        except (ValueError, IndexError):
            bad_rows.add(1)                              # count malformed rows

def has_session(e):
    return e.session != ""

def is_purchase(e):
    return e.event_type == "purchase"

def event_type_and_one(e):
    return (e.event_type, 1)

def is_known_brand(e):
    return e.brand != "unknown"

def brand_and_price(e):
    return (e.brand, e.price)

def get_price(e):
    return e.price

def minus_second_item(pair):
    return -pair[1]                                      # sort from largest to smallest

events = raw_rdd.mapPartitions(parse_partition).filter(has_session)
events.cache()
print("Parsed events:", events.count(), "| bad rows:", bad_rows.value)

# Events per type
print("Events per type:", dict(events.map(event_type_and_one).reduceByKey(add).collect()))

# Revenue and purchases
purchases_rdd = events.filter(is_purchase)
print(f"Purchases: {purchases_rdd.count():,} | revenue: {purchases_rdd.map(get_price).sum():,.2f}")

# Revenue by brand
brand_revenue = purchases_rdd.filter(is_known_brand).map(brand_and_price).reduceByKey(add)
print("\nTop 5 brands by revenue (RDD):")
for brand, revenue in brand_revenue.takeOrdered(5, key=minus_second_item):
    print(f"  {brand:<10} {revenue:>12,.2f}")

print("\nTop 5 brands by revenue (DataFrame, for comparison):")
top_brands.show(5)

**What we did and why**

- `sc.textFile(...)` reads the file as plain text: **one RDD element per line**. If `textFile` fails on your computer (this happens on some Windows setups), the `try / except` loads the same lines with `spark.read.text(...)` instead, so the rest still works.
- `mapPartitions(parse_partition)` calls our function **once per partition** (one chunk of the file) and gives it all the lines of that chunk. Inside, `csv.reader` splits each line into columns safely, and `yield` hands back one `Event` record at a time. An `Event` is a `namedtuple`, a light record whose fields are read by name (`e.price`).
- `bad_rows` is an **accumulator**: a shared counter that workers can only add to and the driver reads with `.value`. It counts lines that could not be parsed (here: none).
- `filter(has_session)` drops events without a session id, and `cache()` keeps the parsed events in memory so the next steps are fast.
- **Counting the classic RDD way:** turn every event into a `(key, 1)` pair and call `reduceByKey(add)`, which adds up the numbers for each key. It combines inside each partition first, so little data travels between workers.
- Revenue by brand works the same way: keep purchases, make `(brand, price)` pairs and add the prices per brand. `takeOrdered(5, key=minus_second_item)` returns the 5 largest (the minus sign flips the order).
- The last lines print the DataFrame answer next to the RDD answer. **Matching numbers confirm both methods are correct.**

> **Key finding:** The RDD result for revenue by brand matches the DataFrame result exactly (MSI 643,492.34, Gigabyte 556,183.04, Palit 484,210.58), which validates the DataFrame analysis. The RDD total revenue is 5,125,395.62 from 37,346 purchases, against 5,125,113.92 from 37,343 purchases in the DataFrame pipeline. The tiny difference (0.006%) exists because the RDD parser keeps some exact duplicate rows (including 3 duplicate purchases) that the DataFrame cleaning removes. The histogram shows that most purchases are small: 8,755 below 25, and only 66 above 1,000.

## Step 22: RDD part 2: customer journeys, broadcast variable and back to a DataFrame

In [ ]:
# ---------- Customer journeys ----------
def session_and_event(e):
    return (e.session, (e.event_time, e.event_type))

def event_types_in_time_order(items):
    """items = all (time, type) pairs of one session. Sort by time and keep only the types."""
    return [event_type for _, event_type in sorted(items)]

def path_text_and_one(pair):
    return (">".join(pair[1]), 1)                        # e.g. view>cart>purchase

def is_short_path(pair):
    return pair[0].count(">") < 3

session_paths = events.map(session_and_event).groupByKey().mapValues(event_types_in_time_order)
session_paths.cache()
print("Sessions:", session_paths.count())
print("Most common short journeys:")
for path, count in session_paths.map(path_text_and_one).filter(is_short_path).reduceByKey(add).takeOrdered(5, key=minus_second_item):
    print(f"  {path:<22} {count:>8,}")

# ---------- Engagement score with a broadcast variable ----------
weights = sc.broadcast({"view": 1, "cart": 3, "purchase": 5})

def user_and_points(e):
    return (e.user_id, weights.value[e.event_type])

user_score = events.map(user_and_points).reduceByKey(add)
print("\nTop 5 most engaged users (user_id, points):")
for user_id, points in user_score.takeOrdered(5, key=minus_second_item):
    print(f"  {user_id}  {points}")

# ---------- Convert the RDD result back to a DataFrame ----------
score_schema = T.StructType([
    T.StructField("user_id", T.LongType(), False),
    T.StructField("engagement_score", T.IntegerType(), False),
])
score_df = spark.createDataFrame(user_score, score_schema)
score_df.orderBy(F.desc("engagement_score")).show(5)
weights.unpersist()

**What we did and why**

- **Customer journeys.** We turn each event into `(session, (time, type))`. `groupByKey()` collects all events of a session in one place, and `mapValues(...)` sorts them by time and keeps only the event types, for example `view, cart, purchase`. Joining them with `>` gives a readable journey text, and `reduceByKey(add)` counts how often each journey happens. (`groupByKey` is the right tool here because we need *all* events of a session together. For plain totals, prefer `reduceByKey`.)
- **Broadcast variable.** `sc.broadcast(...)` sends a small lookup table (view = 1 point, cart = 3, purchase = 5) **once to every worker**. Each worker reads it with `weights.value`. Adding the points per user with `reduceByKey` gives an engagement score. `unpersist()` frees the broadcast when we are done.
- **Back to a DataFrame.** `spark.createDataFrame(rdd, schema)` converts the RDD result into a DataFrame, so we can sort and print it like everything else. A common pattern is: RDD for custom processing, DataFrame for analysis and output.

> **Key finding:** The RDD journeys match the DataFrame result: about 71% of sessions (roughly 348,000 of 490,000) are a single product view, followed by `view>view` (about 60,000) and `view>view>view` (about 18,500). The most common complete buying journey is `view>cart>purchase` (about 5,700 sessions). The engagement score ranks the most active users by points.

# Part 6: Saving the Results

## Step 23: Write Parquet (partitioned), CSV and JSON, then read back

In [ ]:
import shutil
OUTPUT_DIR = "output"
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)       # start with a clean output folder

# Partitioned Parquet: one folder per month
(clean.drop("brand_upper", "brand_title", "brand_len")
      .write.mode("overwrite").partitionBy("year_month")
      .parquet(f"{OUTPUT_DIR}/events_parquet"))

# Small CSV and JSON reports (coalesce(1) = write a single file)
daily.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{OUTPUT_DIR}/daily_revenue_csv")
rev_by_category.coalesce(1).write.mode("overwrite").json(f"{OUTPUT_DIR}/revenue_by_category_json")

print(sorted(os.listdir(f"{OUTPUT_DIR}/events_parquet")))

# Read the Parquet back
parquet_df = spark.read.parquet(f"{OUTPUT_DIR}/events_parquet")
print("Parquet rows :", parquet_df.count())
print("Schema kept  :", parquet_df.dtypes[:4], "...")

# Read the CSV and JSON back
spark.read.option("header", True).option("inferSchema", True).csv(f"{OUTPUT_DIR}/daily_revenue_csv").show(3)
spark.read.json(f"{OUTPUT_DIR}/revenue_by_category_json").show(3)

**What we did and why**

- `df.write.mode(...)` decides what happens if the output already exists: `overwrite`, `append`, `ignore` or `error` (the default). `partitionBy("year_month")` creates **one sub-folder per value**, such as `year_month=2021-01/`.
- **Parquet** is a compressed, column-based file format that **stores its own schema**, so reading it back needs no schema and no parsing. It is the best default for analysis. CSV and JSON are handy for sharing with other tools.
- `coalesce(1)` merges the output into a single file, fine for small reports. Normally Spark writes **one file per partition** plus small `_SUCCESS` marker files.
- For CSV, `inferSchema` costs an extra pass over the data and can guess wrongly, so an explicit schema is better.
- Other formats: ORC, Avro and Delta (need extra packages) and JDBC for databases.

> **Key finding:** The cleaned data is stored as Parquet partitioned by month (6 folders, 884,312 rows). The row count is preserved and the schema is stored in the files.

# Part 7: Predicting Purchases with Machine Learning

**Business goal:** flag carts that are unlikely to convert so the store can react (reminder email, discount). **Problem type:** classification into purchased (1) or abandoned (0).

**Key terms:** a *feature* is an input number the model learns from (for example the number of views), the *label* is the answer we predict, *training* means learning from known examples and *testing* means measuring the model on sessions it has never seen.

## Step 24: Build the session-level dataset

In [ ]:
# FEATURES: one row per session, built only from view and cart events (no purchases!)
features = (
    clean.filter("event_type != 'purchase'")
    .groupBy("user_session")
    .agg(
        F.sum((F.col("event_type") == "view").cast("int")).alias("n_views"),     # number of views
        F.sum((F.col("event_type") == "cart").cast("int")).alias("n_carts"),     # number of cart events
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.max("price").alias("max_price"),
        F.countDistinct("product_id").alias("n_products"),
        F.countDistinct("brand").alias("n_brands"),
        (F.unix_timestamp(F.max("event_ts")) - F.unix_timestamp(F.min("event_ts"))).alias("duration_secs"),
        F.max(F.col("is_weekend").cast("int")).alias("is_weekend"),
    )
    .filter("n_carts >= 1")                         # only sessions that reached the cart
)

# LABEL: did the session contain a purchase? (1.0 = yes, 0.0 = no)
labels = (
    clean.groupBy("user_session")
    .agg(F.max((F.col("event_type") == "purchase").cast("double")).alias("label"))
)

# Attach the label to the features
ml_df = features.join(labels, "user_session").fillna(0)
ml_df.cache()
print("Sessions with a cart:", ml_df.count())
ml_df.groupBy("label").count().show(title="Class balance (1.0 = purchased)")
ml_df.show(5, title="Sample of the modelling dataset")

**What we did and why**

- Features come **only from `view` and `cart` events**. If we included purchase events the model would simply "see the answer". That mistake is called **data leakage**.
- Each session becomes one row: counts, price statistics, number of distinct products and brands, duration and a weekend flag.
- The label (1.0 = purchased, 0.0 = abandoned) is calculated separately with `max(is_purchase)` and attached with a join.
- The class-balance table shows whether the two classes are roughly even. This matters when choosing a metric, which is why we report AUC, accuracy, precision, recall and F1.

> **Key finding:** The modelling dataset has **41,270 cart sessions, almost perfectly balanced** (20,762 purchased against 20,508 abandoned). Accuracy is therefore a fair metric next to AUC.

## Step 25: Train and compare six Spark MLlib algorithms

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import (LogisticRegression, LinearSVC, NaiveBayes,
                                       DecisionTreeClassifier, RandomForestClassifier, GBTClassifier)
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# The input columns the models will learn from
feature_cols = ["n_views", "n_carts", "avg_price", "max_price", "n_products", "n_brands", "duration_secs", "is_weekend"]

# 80% of sessions to train, 20% to test
train, test = ml_df.randomSplit([0.8, 0.2], seed=42)
train.cache(); test.cache()
print(f"Train rows: {train.count():,} | Test rows: {test.count():,}")

# Preprocessing stages
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")      # pack columns into one vector
scaler    = StandardScaler(inputCol="raw_features", outputCol="scaled", withMean=True, withStd=True)   # put features on the same scale

# The six algorithms: name -> (model, does it need scaled features?)
candidates = {
    "Logistic Regression":    (LogisticRegression(featuresCol="scaled", maxIter=50, regParam=0.01), True),
    "Linear SVM":             (LinearSVC(featuresCol="scaled", maxIter=50, regParam=0.01), True),
    "Naive Bayes":            (NaiveBayes(featuresCol="raw_features", smoothing=1.0), False),
    "Decision Tree":          (DecisionTreeClassifier(featuresCol="raw_features", maxDepth=6, seed=42), False),
    "Random Forest":          (RandomForestClassifier(featuresCol="raw_features", numTrees=60, maxDepth=7, seed=42), False),
    "Gradient Boosted Trees": (GBTClassifier(featuresCol="raw_features", maxIter=30, maxDepth=5, seed=42), False),
}

# Evaluators measure how good predictions are
auc_eval = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

def mc(metric):
    """Build an evaluator for accuracy / precision / recall / f1."""
    return MulticlassClassificationEvaluator(labelCol="label", metricName=metric)

spark_models, spark_rows = {}, []
for name, (estimator, scaled) in candidates.items():
    # Pipeline = assembler, (scaler if needed), then the model
    if scaled:
        stages = [assembler, scaler, estimator]
    else:
        stages = [assembler, estimator]

    t0 = time.time()
    model = Pipeline(stages=stages).fit(train)          # TRAIN
    fit_secs = time.time() - t0

    pred = model.transform(test)                        # PREDICT on unseen data
    spark_models[name] = model
    spark_rows.append({
        "Algorithm": name,
        "AUC": auc_eval.evaluate(pred),
        "Accuracy": mc("accuracy").evaluate(pred),
        "Precision": mc("weightedPrecision").evaluate(pred),
        "Recall": mc("weightedRecall").evaluate(pred),
        "F1": mc("f1").evaluate(pred),
        "Train time (s)": fit_secs,
    })

# Results table, best AUC first
spark_results = pd.DataFrame(spark_rows).sort_values("AUC", ascending=False).reset_index(drop=True)
show_pdf(spark_results, "Spark MLlib: test-set performance", highlight_max=["AUC", "Accuracy", "Precision", "Recall", "F1"])

**What we did and why**

- A **Pipeline** chains the steps *assemble features, (scale), train model*. Training and prediction then always apply exactly the same preprocessing.
- `VectorAssembler` packs the feature columns into one column holding a vector, which is the format Spark ML models need. `StandardScaler` puts features on a similar scale.
- **Logistic regression and linear SVM** are sensitive to scale, so they use the scaled features. **Tree models** do not need scaling. **Naive Bayes** needs non-negative numbers, so it uses the raw features (all counts, prices and durations).
- All six models use the same train/test split, so the comparison is fair. We `cache` both parts because they are used six times.
- The loop trains each model, predicts on the test set and records AUC, accuracy, precision, recall, F1 and training time. The table highlights the best value of each column in green.
- **AUC** (area under the ROC curve) measures how well a model ranks buyers above non-buyers: 0.5 is random guessing, 1.0 is perfect. It is our main metric because it does not depend on a probability cut-off.

> **Key finding:** The best Spark MLlib models are **Gradient Boosted Trees (AUC 0.67, accuracy 0.62)** and Random Forest (AUC 0.66). Simple models (Logistic Regression 0.59, Naive Bayes 0.53) are clearly weaker, which means the relationship is non-linear.

## Step 26: Feature importance: what drives a purchase?

In [ ]:
# Start a table that has one row per feature
importance = pd.DataFrame({"Feature": feature_cols})

# Add one column of importances per tree model.
# stages[-1] is the last pipeline stage, which is the trained model itself.
for name in ["Decision Tree", "Random Forest", "Gradient Boosted Trees"]:
    importance[name] = spark_models[name].stages[-1].featureImportances.toArray()

importance = importance.sort_values("Random Forest", ascending=False).reset_index(drop=True)
show_pdf(importance, "Feature importance (each column sums to 1)", highlight_max=["Decision Tree", "Random Forest", "Gradient Boosted Trees"])

# Logistic regression coefficients (direction and strength of each feature)
lr_coefs = spark_models["Logistic Regression"].stages[-1].coefficients.toArray()
coef_df = pd.DataFrame({"Feature": feature_cols, "Coefficient (scaled)": lr_coefs})
# sort by the size of the coefficient (ignore the + or - sign)
coef_df = coef_df.reindex(coef_df["Coefficient (scaled)"].abs().sort_values(ascending=False).index).reset_index(drop=True)
show_pdf(coef_df, "Logistic regression coefficients")

**What we did and why**

- `featureImportances` says **how much each feature helped** the trees make good splits. A higher number means a more useful feature. Comparing three tree models shows which signals are stable.
- Logistic regression **coefficients** show direction as well as strength: a positive number pushes towards "purchase", a negative number towards "abandon". Because the features were scaled, the numbers can be compared directly.
- `stages[-1]` picks the last stage of the pipeline, which is the trained classifier.

> **Key finding:** **Session duration is the most important feature** (importance 0.32 to 0.46), followed by the number of cart events and the maximum price. Day of the week (weekend) matters very little.

## Step 27: Train eight scikit-learn algorithms on the same split

In [ ]:
import pickle, json
import numpy as np
from sklearn import ensemble, linear_model, naive_bayes, neighbors, tree
from sklearn import metrics as skm
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler as SkScaler

# After aggregation the table is small, so we can move it from Spark to pandas
train_pdf = train.toPandas()
test_pdf  = test.toPandas()
X_train, y_train = train_pdf[feature_cols], train_pdf["label"].astype(int)     # X = inputs, y = answers
X_test,  y_test  = test_pdf[feature_cols],  test_pdf["label"].astype(int)
print(f"pandas train: {X_train.shape} | test: {X_test.shape}")

# The eight scikit-learn algorithms
sk_models = {
    "Logistic Regression":   make_pipeline(SkScaler(), linear_model.LogisticRegression(max_iter=1000)),
    "K-Nearest Neighbors":   make_pipeline(SkScaler(), neighbors.KNeighborsClassifier(n_neighbors=25)),
    "Gaussian Naive Bayes":  naive_bayes.GaussianNB(),
    "Decision Tree":         tree.DecisionTreeClassifier(max_depth=6, random_state=42),
    "Random Forest":         ensemble.RandomForestClassifier(n_estimators=200, max_depth=10, n_jobs=-1, random_state=42),
    "Extra Trees":           ensemble.ExtraTreesClassifier(n_estimators=200, max_depth=12, n_jobs=-1, random_state=42),
    "Gradient Boosting":     ensemble.GradientBoostingClassifier(n_estimators=150, max_depth=3, random_state=42),
    "Hist Gradient Boosting": ensemble.HistGradientBoostingClassifier(max_iter=200, random_state=42),
}

sk_rows = []
for name, model in sk_models.items():
    t0 = time.time()
    model.fit(X_train, y_train)                      # TRAIN
    fit_secs = time.time() - t0

    proba = model.predict_proba(X_test)[:, 1]        # probability of "purchase"
    pred  = model.predict(X_test)                    # predicted class (0 or 1)
    sk_rows.append({
        "Algorithm": name,
        "AUC": skm.roc_auc_score(y_test, proba),
        "Accuracy": skm.accuracy_score(y_test, pred),
        "Precision": skm.precision_score(y_test, pred, zero_division=0),
        "Recall": skm.recall_score(y_test, pred, zero_division=0),
        "F1": skm.f1_score(y_test, pred, zero_division=0),
        "Train time (s)": fit_secs,
    })

sk_results = pd.DataFrame(sk_rows).sort_values("AUC", ascending=False).reset_index(drop=True)
show_pdf(sk_results, "scikit-learn: test-set performance", highlight_max=["AUC", "Accuracy", "Precision", "Recall", "F1"])

**What we did and why**

- Spark did the heavy work of building the dataset from 885K events. After summarising to one row per session, the table is small enough to move to pandas with `toPandas()`. This is a very common pattern: **Spark for big data preparation, scikit-learn for model training**.
- We convert the **same** `train` and `test` tables, so scikit-learn is tested on exactly the same rows as Spark MLlib.
- `X` holds the input columns and `y` holds the answers. `model.fit(X, y)` trains, `predict_proba` gives purchase probabilities and `predict` gives 0 or 1.
- Eight model families are compared: linear (logistic regression), distance-based (KNN), probability-based (Gaussian Naive Bayes), a single tree, and four ensembles (Random Forest, Extra Trees, Gradient Boosting, Hist Gradient Boosting). Models that need scaling are wrapped in `make_pipeline(StandardScaler(), ...)`, so the scaler is stored inside the model.
- The loop records the same metrics as the Spark table, so the two tables can be compared directly.

> **Key finding:** scikit-learn reaches the same level as Spark: **Gradient Boosting and Hist Gradient Boosting both have AUC 0.67**. Hist Gradient Boosting trains in only 0.33 s. Gaussian Naive Bayes fails to find purchasers (recall 0.04).

## Step 28: Save every scikit-learn model as a .pkl file

In [ ]:
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)       # create the folder if it does not exist

def slug(name):
    """Turn 'Random Forest' into 'random_forest' for use in file names."""
    return name.lower().replace(" ", "_").replace("-", "_")

saved = []
for name, model in sk_models.items():
    path = os.path.join(MODEL_DIR, f"{slug(name)}.pkl")
    with open(path, "wb") as f:              # "wb" = write binary
        pickle.dump(model, f)                # save the trained model
    saved.append({"Model": name, "File": path, "Size (KB)": os.path.getsize(path) / 1024})

# Extra information needed to use the models later
with open(os.path.join(MODEL_DIR, "feature_columns.json"), "w") as f:
    json.dump(feature_cols, f, indent=2)
sk_results.to_csv(os.path.join(MODEL_DIR, "model_metrics.csv"), index=False)

# Spark pipelines cannot be pickled, so save the best one in Spark's own format (a folder)
best_spark_name = spark_results.iloc[0]["Algorithm"]
spark_models[best_spark_name].write().overwrite().save(os.path.join(MODEL_DIR, f"spark_{slug(best_spark_name)}"))

show_pdf(pd.DataFrame(saved), "Saved .pkl model files")
print("Also saved: feature_columns.json, model_metrics.csv and the Spark pipeline folder:", f"spark_{slug(best_spark_name)}")

**What we did and why**

- `pickle.dump(model, file)` saves a trained scikit-learn model into a `.pkl` file. Pipelines that include a scaler are saved as **one object**, so preprocessing and model always travel together.
- `feature_columns.json` records the exact input column order the models expect. `model_metrics.csv` stores the test results for documentation.
- **Why Spark models are not saved as `.pkl`:** a Spark ML model is a wrapper around an object living in the Java engine (JVM), which Python's `pickle` cannot save. Spark has its own writer, `model.write().save(path)`, which creates a folder. `PipelineModel.load(path)` brings it back.
- Only load `.pkl` files from sources you trust, because loading a pickle can run arbitrary code.

> **Key finding:** All eight scikit-learn models are stored as `.pkl` files (from 1 KB for Naive Bayes to 16 MB for Random Forest) together with the feature list, the metrics and the best Spark pipeline.

## Step 29: Load a saved model and score new sessions

In [ ]:
# Load the best scikit-learn model from disk
best_sk_name = sk_results.iloc[0]["Algorithm"]
with open(os.path.join(MODEL_DIR, f"{slug(best_sk_name)}.pkl"), "rb") as f:    # "rb" = read binary
    loaded = pickle.load(f)

# 1) The reloaded model must give exactly the same predictions as the one in memory
same = np.array_equal(loaded.predict(X_test), sk_models[best_sk_name].predict(X_test))
print(f"Best scikit-learn model: {best_sk_name}")
print("Reloaded model gives identical predictions:", same)

# 2) Score three made-up sessions
with open(os.path.join(MODEL_DIR, "feature_columns.json")) as f:
    cols = json.load(f)

new_sessions = pd.DataFrame([
    {"n_views": 1,  "n_carts": 1, "avg_price": 25.0,  "max_price": 25.0,  "n_products": 1, "n_brands": 1, "duration_secs": 60,   "is_weekend": 0},
    {"n_views": 12, "n_carts": 2, "avg_price": 180.0, "max_price": 450.0, "n_products": 8, "n_brands": 4, "duration_secs": 1500, "is_weekend": 1},
    {"n_views": 40, "n_carts": 5, "avg_price": 320.0, "max_price": 900.0, "n_products": 20, "n_brands": 9, "duration_secs": 5400, "is_weekend": 0},
])[cols]                                              # [cols] puts the columns in training order

scored = new_sessions.assign(
    purchase_probability=loaded.predict_proba(new_sessions)[:, 1],
    prediction=np.where(loaded.predict(new_sessions) == 1, "will purchase", "will abandon"),
)
show_pdf(scored, "Scoring new sessions with the loaded .pkl model")

# 3) Reload the saved Spark pipeline and score the test set again
from pyspark.ml import PipelineModel
spark_loaded = PipelineModel.load(os.path.join(MODEL_DIR, f"spark_{slug(best_spark_name)}"))
spark_auc = auc_eval.evaluate(spark_loaded.transform(test))
print(f"Reloaded Spark pipeline ({best_spark_name}) test AUC: {spark_auc:.4f}")

**What we did and why**

- `pickle.load` brings the model back from disk. Comparing `predict` results with `np.array_equal` proves that saving and loading changed nothing. This is a good habit for every saved model.
- To use a model in production you only need the `.pkl` file and `feature_columns.json`. The `[cols]` part re-orders the input columns so they always match the training order.
- The three made-up sessions (a quick one-item visit, a medium session, a long multi-product session) show how the model turns behaviour into a **purchase probability**. A shop could, for example, email a reminder to carts above a chosen probability.
- The last lines reload the saved Spark pipeline with `PipelineModel.load` and score the test set again, confirming that Spark's own save format works too.

> **Key finding:** The reloaded model gives identical predictions. It scores a long, multi-product session at **67% purchase probability** and a medium session with a high price at 41%, so it can rank carts by purchase likelihood.

## Step 30: Charts: business results and model comparison

In [ ]:
import matplotlib.pyplot as plt

# A clean chart style
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 12,
                     "axes.spines.top": False, "axes.spines.right": False})

# ---------- 1) Bring only SMALL summary tables to pandas ----------
monthly_pdf = monthly.toPandas().sort_values("year_month")             # from the monthly trend step

weekday_pdf = (clean.groupBy("dow", "day_name")
                    .agg(F.sum("revenue").alias("revenue"))
                    .orderBy("dow").toPandas())

tier_order = ["budget", "mid", "high", "premium"]
tier_pdf = (clean.filter("event_type = 'purchase'").groupBy("price_tier").count().toPandas()
                 .set_index("price_tier").reindex(tier_order).reset_index())

segment_pdf = (rfm.groupBy("segment")
                  .agg(F.count("*").alias("customers"), F.round(F.avg("monetary"), 2).alias("avg_spend"))
                  .toPandas().sort_values("customers"))

# ---------- 2) Six business charts in a 2 x 3 grid ----------
fig, ax = plt.subplots(2, 3, figsize=(18, 10))

# (a) Purchases per month (bars) and purchase rate (line)
ax[0, 0].bar(monthly_pdf["year_month"], monthly_pdf["purchase"], color="#1f4e79")
ax[0, 0].set_title("Purchases per month")
ax[0, 0].tick_params(axis="x", rotation=45)
rate_axis = ax[0, 0].twinx()                                           # second y-axis for the rate
rate_axis.plot(monthly_pdf["year_month"], monthly_pdf["purchase_rate_pct"], color="#e67e22", marker="o")
rate_axis.set_ylabel("Purchase rate (% of views)")
rate_axis.grid(False)

# (b) Revenue by weekday
ax[0, 1].bar(weekday_pdf["day_name"], weekday_pdf["revenue"] / 1000, color="#2e86c1")
ax[0, 1].set_title("Revenue by weekday (thousands)")

# (c) Purchases by price tier
ax[0, 2].bar(tier_pdf["price_tier"], tier_pdf["count"], color=["#85c1e9", "#2e86c1", "#1f4e79", "#154360"])
ax[0, 2].set_title("Purchases by price tier")

# (d) Customers per RFM segment
ax[1, 0].barh(segment_pdf["segment"], segment_pdf["customers"], color="#27ae60")
ax[1, 0].set_title("Customers per RFM segment")

# (e) Average spend per RFM segment
ax[1, 1].barh(segment_pdf["segment"], segment_pdf["avg_spend"], color="#f39c12")
ax[1, 1].set_title("Average spend per RFM segment")

# (f) Feature importance from the Random Forest (from the feature importance step)
imp = importance.sort_values("Random Forest")
ax[1, 2].barh(imp["Feature"], imp["Random Forest"], color="#8e44ad")
ax[1, 2].set_title("What drives a purchase? (feature importance)")

fig.suptitle("Business Results at a Glance", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

# ---------- 3) Model comparison: every Spark and scikit-learn model by AUC ----------
all_results = pd.concat([
    spark_results.assign(Framework="Spark MLlib"),
    sk_results.assign(Framework="scikit-learn"),
]).sort_values("AUC")

bar_colors = all_results["Framework"].map({"Spark MLlib": "#e67e22", "scikit-learn": "#1f4e79"})
bar_labels = all_results["Algorithm"] + "  (" + all_results["Framework"] + ")"

fig, ax2 = plt.subplots(figsize=(11, 7))
bars = ax2.barh(bar_labels, all_results["AUC"], color=bar_colors)
ax2.axvline(0.5, color="grey", linestyle="--", linewidth=1)            # 0.5 = random guessing
ax2.set_xlim(0.45, max(0.8, all_results["AUC"].max() + 0.05))
ax2.set_xlabel("Test AUC (0.5 = random guessing)")
ax2.set_title("Model comparison: predicting purchase from a cart session", fontweight="bold")
for bar, value in zip(bars, all_results["AUC"]):                       # write the AUC next to each bar
    ax2.text(value + 0.002, bar.get_y() + bar.get_height() / 2, f"{value:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

best = all_results.sort_values("AUC", ascending=False).iloc[0]
print(f"Best model: {best['Algorithm']} ({best['Framework']}) with AUC = {best['AUC']:.4f}")

**What we did and why**

Charts make results easy to explain, so this step turns the main tables into pictures.

- **Summarise in Spark, plot in pandas.** We never plot the raw 885K rows. We first make small summary tables in Spark (a few rows each) and only then convert them with `toPandas()`.
- **Chart (a)** shows purchases per month as bars and the purchase rate as a line. `twinx()` adds a second y-axis on the right so both can share one chart.
- **Charts (b) and (c)** show revenue by weekday and purchases by price tier. We fix the tier order (`budget, mid, high, premium`) with `reindex`.
- **Charts (d) and (e)** show how many customers each RFM segment has and how much each segment spends on average.
- **Chart (f)** shows which features matter most to the Random Forest model.
- `plt.subplots(2, 3)` creates a grid of 2 rows and 3 columns, and `ax[row, column]` picks one chart in the grid.
- The **last chart** ranks all 14 models by AUC. Orange bars are Spark MLlib, blue bars are scikit-learn, and the dashed line at 0.5 marks random guessing. Any bar to the right of that line has learned something useful.

> **Key finding:** The charts tell the whole story at a glance: purchases and the purchase rate rise month after month, revenue is spread evenly over the week (Monday highest, Saturday lowest), mid-priced and high-priced items make up most purchases, the **lapsed segment is the largest but Champions spend the most per customer**, session duration is the strongest signal for the model, and the boosting models are the best (AUC about 0.67).

# Part 8: Conclusions

## Step 31: Release resources and stop Spark

In [ ]:
ml_df.unpersist()                 # free the cached modelling table
events.unpersist()                # free the cached RDD
clean.unpersist()                 # free the cached cleaned DataFrame
spark.catalog.clearCache()        # remove anything else still cached
spark.stop()                      # shut Spark down
print("Spark session stopped.")

**What we did and why**

- `unpersist()` releases data we cached earlier, and `spark.catalog.clearCache()` removes everything cached in the session.
- `spark.stop()` shuts Spark down and frees memory, workers and the Spark web page port. In notebooks and scripts it is good practice to **always stop the session when you are done**.

## Final Conclusions

1. **Data quality is high.** Only 0.09% of rows were removed. Missing brands and categories (about a quarter of events) were kept as `unknown`.
2. **Revenue is concentrated.** Computers earn 72.8% of the 5.13 million revenue, driven by PC components from MSI, Gigabyte and Palit.
3. **The business is growing through higher spend per user.** Monthly revenue rose from 563K to 1.38M and revenue per user almost tripled, while monthly active users stayed flat.
4. **The funnel leaks at the cart.** 71% of sessions are single-view visits, only 8.45% of sessions with a view reach the cart and 49.7% of carts are abandoned.
5. **Customers shop in the morning (UTC) and on weekdays.** Conversion peaks between 09:00 and 11:00 UTC.
6. **A few customers carry the business.** About 29% of buyers produce about 61% of revenue, and the largest segment (10,652 customers) has gone quiet for about 113 days on average.
7. **Purchases are only partly predictable from browsing.** The best models reach AUC 0.67 on a balanced problem, and session duration is the strongest signal.
8. **The pipeline is production-minded.** It uses an explicit schema, caching, Parquet with partition pruning and an independent RDD cross-check (revenue differs by only 0.006%).

## Business Recommendations

| Priority | Recommendation | Evidence |
|---|---|---|
| 1 | **Recover abandoned carts** with reminder emails or small incentives within the same day | 49.7% cart abandonment, 41,270 cart sessions |
| 2 | **Protect and grow computer components**: secure stock and promote MSI, Gigabyte and Palit video cards | 72.8% of revenue |
| 3 | **Run campaigns at 09:00 to 11:00 UTC** and schedule maintenance for 00:00 to 03:00 UTC | Highest conversion and traffic in the morning |
| 4 | **Win back lapsed customers** and reward Champions with a loyalty programme | 10,652 lapsed customers, Champions spend 844 on average |
| 5 | **Improve the long tail of the catalogue** with better pages, search and bundles around high-converting cheap products | Only 12% of viewed products were ever bought |
| 6 | **Score every cart with the saved model** and target low-probability carts | Gradient Boosting, AUC 0.67 |
| 7 | **Capture brand and category data** at the source | 24% to 27% missing |

## Limitations

- The data covers only five months, so seasonality over a full year cannot be assessed. September 2020 is a partial month.
- Times are in UTC, not the shop's local time, so the hour of day is a relative pattern.
- The model uses only browsing behaviour from before the purchase decision. Its moderate AUC (0.67) is expected because features like customer history, product category and discount information are not included.
- The dataset has no information about marketing campaigns, stock levels or discounts, so the reasons behind the January 2021 jump can only be guessed and not proven.
- Revenue means the listed price of purchase events, not net revenue after returns or costs.

## Next Steps

- Add features (time of day, product category, user history) and tune the best models with cross-validation.
- Build a recommendation model (ALS) from user-product interactions.
- Use Structured Streaming to process the events as a live stream.
- Store the cleaned data in a Delta Lake table for incremental loads.
- Publish the dashboard as an interactive report (for example Streamlit or Power BI) on top of the Parquet output.